# 03 — Clean the data and split it by time

Full dataset (not the 10% sample): `data/interim/` → `data/processed/{train,val,test,items}.parquet`.

**Leakage rules**
- Every feature uses only data from before the cutoff.
- Fit scalers, encoders, TF-IDF and K-Means on train only; `.transform()` val and test.
- Never use metadata `average_rating` / `rating_number` (collected in 2023, they include future reviews). Use the `train_*` columns recomputed below.
- Never shuffle rows across time.

In [1]:
import html
import re
from pathlib import Path

import numpy as np
import pandas as pd

START = pd.Timestamp("2018-01-01")
T1 = pd.Timestamp("2022-07-01")  # validation starts
T2 = pd.Timestamp("2023-01-01")  # test starts
K = 5

OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

reviews = pd.read_parquet("../data/interim/reviews.parquet")
items = pd.read_parquet("../data/interim/items.parquet")
counts = {"raw": len(reviews)}
print(reviews.shape, items.shape)

(4624615, 8) (137269, 9)


## 1 — Clean reviews

In [2]:
reviews = reviews.dropna(subset=["user_id", "parent_asin", "rating"])
counts["after dropna"] = len(reviews)

reviews = reviews.drop_duplicates()
counts["after exact dedup"] = len(reviews)

reviews = (reviews.sort_values("timestamp", kind="stable")
                  .drop_duplicates(["user_id", "parent_asin"], keep="last"))
counts["after keep-latest per (user, product)"] = len(reviews)

## 3 / 4 — Define an interaction, keep a recent window

In [3]:
reviews["liked"] = (reviews["rating"] >= 4).astype("int8")
reviews = reviews[reviews["timestamp"] >= START]
counts[f"from {START.date()}"] = len(reviews)

## 5 — 5-core filter

In [4]:
def k_core(df, k=5):
    while True:
        user_counts = df["user_id"].map(df["user_id"].value_counts())
        item_counts = df["parent_asin"].map(df["parent_asin"].value_counts())
        keep = (user_counts >= k) & (item_counts >= k)
        if keep.all():
            return df
        df = df[keep]

reviews = k_core(reviews, K)
counts[f"{K}-core"] = len(reviews)

assert reviews.groupby("user_id").size().min() >= K
assert reviews.groupby("parent_asin").size().min() >= K
print(pd.Series(counts).to_string())
print(f"users {reviews['user_id'].nunique():,}  products {reviews['parent_asin'].nunique():,}  "
      f"liked {reviews['liked'].mean():.1%}")

raw                                      4624615
after dropna                             4624615
after exact dedup                        4570966
after keep-latest per (user, product)    4555500
from 2018-01-01                          2374766
5-core                                    184567
users 24,243  products 9,999  liked 83.3%


## 6 — Split by time

The guide suggests T1 = 2023-06-01, T2 = 2023-08-01, but after the 5-core filter June–September 2023 holds only a few hundred reviews (collection ended Sept 2023, so the last months are nearly empty). T1 = 2022-07-01 and T2 = 2023-01-01 (chosen in Phase 2) give usable validation and test sets, each with one holiday peak.

In [5]:
print(reviews.set_index("timestamp").resample("MS").size().loc["2022-01":].to_string())

train = reviews[reviews["timestamp"] < T1]
val = reviews[(reviews["timestamp"] >= T1) & (reviews["timestamp"] < T2)]
test = reviews[reviews["timestamp"] >= T2]

assert len(train) + len(val) + len(test) == len(reviews)
assert train["timestamp"].max() < T1 and val["timestamp"].max() < T2 and test["timestamp"].min() >= T2

for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5} {len(df):>8,} rows ({len(df) / len(reviews):.1%})  users {df['user_id'].nunique():>6,}  "
          f"products {df['parent_asin'].nunique():>6,}  {df['timestamp'].min().date()} -> {df['timestamp'].max().date()}")

timestamp
2022-01-01    2637
2022-02-01    2120
2022-03-01    2429
2022-04-01    2083
2022-05-01    2095
2022-06-01    1844
2022-07-01    2354
2022-08-01    2824
2022-09-01    2845
2022-10-01    3793
2022-11-01    4995
2022-12-01    4516
2023-01-01    4531
2023-02-01    2930
2023-03-01    3447
2023-04-01    1951
2023-05-01     476
2023-06-01     172
2023-07-01     127
2023-08-01      81
2023-09-01       1
Freq: MS
train  149,524 rows (81.0%)  users 22,324  products  8,519  2018-01-01 -> 2022-06-30
val     21,327 rows (11.6%)  users  6,939  products  4,696  2022-07-01 -> 2022-12-31
test    13,716 rows (7.4%)  users  4,686  products  3,917  2023-01-01 -> 2023-09-02


## 7 — Which users to score

**warm** = seen in train and has ≥1 liked product in the window (main metric). **cold** = has ≥1 liked product in the window but no train history (tracked separately).

In [6]:
def eval_users(window, history_users, split):
    liked_users = pd.Series(window.loc[window["liked"] == 1, "user_id"].unique())
    group = np.where(liked_users.isin(history_users), "warm", "cold")
    return pd.DataFrame({"user_id": liked_users, "split": split, "group": group})

train_users = set(train["user_id"])
users_to_score = pd.concat([
    eval_users(val, train_users, "val"),
    eval_users(test, train_users, "test"),
], ignore_index=True)
print(users_to_score.groupby(["split", "group"]).size().unstack())

group  cold  warm
split            
test   1428  2686
val    1495  4549


## 2 — Clean metadata (+ product rating features from train only)

In [7]:
TAG = re.compile(r"<[^>]+>")
SPACE = re.compile(r"\s+")

def to_text(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, str):
        return v
    return " ".join(str(x) for x in v)

def clean_text(s):
    s = html.unescape(TAG.sub(" ", s))
    return SPACE.sub(" ", s).strip().lower()

it = items[items["parent_asin"].isin(reviews["parent_asin"])].copy()

it["price"] = pd.to_numeric(it["price"], errors="coerce")
it["price_missing"] = it["price"].isna().astype("int8")
median_price = it["price"].median()
it["price"] = it["price"].fillna(median_price)

it["item_text"] = (it["title"].map(to_text) + " " + it["features"].map(to_text) + " "
                   + it["description"].map(to_text)).map(clean_text)
it["store"] = it["store"].fillna("").str.strip().replace("", "unknown")
it["platform"] = it["categories"].map(lambda c: c[1] if c is not None and len(c) > 1 else "unknown")

train_stats = train.groupby("parent_asin").agg(
    train_rating_count=("rating", "size"),
    train_avg_rating=("rating", "mean"),
    train_liked_rate=("liked", "mean"),
)
it = it.drop(columns=["average_rating", "rating_number", "features", "description"]).join(train_stats, on="parent_asin")
it["train_rating_count"] = it["train_rating_count"].fillna(0).astype("int32")
it["train_avg_rating"] = it["train_avg_rating"].fillna(train["rating"].mean())
it["train_liked_rate"] = it["train_liked_rate"].fillna(train["liked"].mean())

assert it["parent_asin"].is_unique and set(reviews["parent_asin"]) <= set(it["parent_asin"])
print(f"median price used for gaps: ${median_price:.2f}  price_missing {it['price_missing'].mean():.1%}  "
      f"store unknown {(it['store'] == 'unknown').mean():.1%}  no train reviews {(it['train_rating_count'] == 0).mean():.1%}")
print(it.dtypes)
it[["parent_asin", "item_text"]].head(3)

median price used for gaps: $27.97  price_missing 24.3%  store unknown 0.2%  no train reviews 14.8%
parent_asin               str
title                     str
price                 float64
store                     str
categories             object
price_missing            int8
item_text                 str
platform                  str
train_rating_count      int32
train_avg_rating      float64
train_liked_rate      float64
dtype: object


,parent_asin,item_text
2,B00Z9TLVK0,nba 2k17 - early tip off edition - playstation...
7,B07H93H878,extremerate soft touch top shell front housing...
10,B07DL353F8,tales of vesperia - definitive edition - plays...


## 9 — Save

In [8]:
keep_cols = ["user_id", "parent_asin", "rating", "liked", "timestamp",
             "verified_purchase", "helpful_vote", "title", "text"]
for name, df in [("train", train), ("val", val), ("test", test)]:
    df[keep_cols].sort_values("timestamp", kind="stable").to_parquet(OUT / f"{name}.parquet", index=False)
it.to_parquet(OUT / "items.parquet", index=False)
users_to_score.to_parquet(OUT / "eval_users.parquet", index=False)

for f in sorted(OUT.glob("*.parquet")):
    print(f"{f.name:20} {len(pd.read_parquet(f)):>8,} rows  {f.stat().st_size / 1e6:6.1f} MB")

eval_users.parquet     10,158 rows     0.2 MB
items.parquet           9,999 rows     8.9 MB
test.parquet           13,716 rows     4.3 MB
train.parquet         149,524 rows    35.1 MB
val.parquet            21,327 rows     7.1 MB
